# Model Inference Diagnostics (Training + Synthetic + Production)

This notebook helps you compare multiple `.keras` models across multiple datasets with the same preprocessing pipeline.

What it gives you:
- prediction class distribution per model/dataset
- confidence statistics
- per-class metrics when labels are present (`activity`)
- support for both labeled and unlabeled datasets

## ⚠️ Model Status Note — pretrained baseline excluded

The pretrained mentor model `models/pretrained/fine_tuned_model_1dcnnbilstm.keras` is included in this notebook **for reference only**. It has been formally excluded from pipeline comparison due to undocumented training provenance.

- Forensic evidence: `docs/pretrained_model_exclusion_report.md`
- Exclusion record: `src/config.py::PRETRAINED_MODEL_STATUS`
- Thesis section: `docs/thesis_section_4x_pretrained_analysis.md`
- Investigation ledger: `PIPELINE_DIAGNOSTIC_CHECKLIST.md`

**Do not use the pretrained model's numbers produced in this notebook for thesis comparison.** The only reproducible baseline is the retrained model trained with the user-grouped holdout protocol (see `scripts/evaluate_user_holdout.py`).

In [ ]:
# CELL 1: Imports + runtime info
# Loads libraries and prints TensorFlow/GPU availability.
import json
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

print('TensorFlow:', tf.__version__)
print('Num GPUs:', len(tf.config.list_physical_devices('GPU')))

In [ ]:
# CELL 2: User configuration
# Set project path, model list, dataset list, and preprocessing controls.
# ---- USER CONFIG ----
PROJECT_ROOT = Path(r"D:/study apply/ML Ops/MasterArbeit_MLops")

# Add/remove models here
MODELS: Dict[str, Path] = {
    'pretrained/fine_tuned_1dcnnbilstm': PROJECT_ROOT / 'models/pretrained/fine_tuned_model_1dcnnbilstm.keras',
    'retrained/har_20260327_223917': PROJECT_ROOT / 'models/retrained/har_model_20260327_223917.keras',
    'retrained/har_20260327_213930': PROJECT_ROOT / 'models/retrained/har_model_20260327_213930.keras',
    'retrained/har_20260219_123616': PROJECT_ROOT / 'models/retrained/har_model_20260219_123616.keras',
}

# Add/remove datasets here
# Each dataset should be a fused CSV with 6 channels and optional activity label.
DATASETS: Dict[str, Path] = {
    'training_labeled': PROJECT_ROOT / 'data/all_users_data_labeled.csv',
    'synthetic_codex': PROJECT_ROOT / 'data/synthetic_runs/synthetic_codex_input.csv',
    'synthetic_other': PROJECT_ROOT / 'data/synthetic_runs/synthetic_other_input.csv',
    'production_example': PROJECT_ROOT / 'data/processed/sensor_fused_50Hz.csv',
}

# Preprocessing controls
SCALER_CONFIG = PROJECT_ROOT / 'data/prepared/config.json'
ENABLE_NORMALIZATION = True
WINDOW_SIZE = 200
WINDOW_STEP = 100  # 50% overlap
BATCH_SIZE = 64

print('Models configured:', len(MODELS))
print('Datasets configured:', len(DATASETS))

In [ ]:
# CELL 3: Preprocessing helpers
# Defines label mapping, schema normalization, unit conversion, z-score, and windowing.
LABEL_MAP = {
    0: 'ear_rubbing',
    1: 'forehead_rubbing',
    2: 'hair_pulling',
    3: 'hand_scratching',
    4: 'hand_tapping',
    5: 'knuckles_cracking',
    6: 'nail_biting',
    7: 'nape_rubbing',
    8: 'sitting',
    9: 'smoking',
    10: 'standing',
}
CLASS_TO_IDX = {v: k for k, v in LABEL_MAP.items()}

SENSOR_COLS = ['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz']
RENAME_MAP = {
    'Ax_w': 'Ax', 'Ay_w': 'Ay', 'Az_w': 'Az',
    'Gx_w': 'Gx', 'Gy_w': 'Gy', 'Gz_w': 'Gz',
    'acc_x': 'Ax', 'acc_y': 'Ay', 'acc_z': 'Az',
    'gyr_x': 'Gx', 'gyr_y': 'Gy', 'gyr_z': 'Gz',
}

def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for old, new in RENAME_MAP.items():
        if old in out.columns:
            out = out.rename(columns={old: new})
    missing = [c for c in SENSOR_COLS if c not in out.columns]
    if missing:
        raise ValueError(f'Missing sensor columns after schema normalization: {missing}')
    return out

def maybe_unit_convert_accel(df: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    out = df.copy()
    accel_cols = ['Ax', 'Ay', 'Az']
    max_abs = float(np.abs(out[accel_cols].values).max())
    # heuristic: large range means milliG
    if max_abs > 100.0:
        factor = 9.80665 / 1000.0
        for c in accel_cols:
            out[c] = out[c].astype(float) * factor
        return out, 'milliG_to_m/s2'
    return out, 'm/s2_no_conversion'

def apply_zscore(df: pd.DataFrame, scaler_config: Path, enabled: bool) -> pd.DataFrame:
    out = df.copy()
    vals = out[SENSOR_COLS].astype(float).values
    if np.isnan(vals).any():
        raise ValueError('NaN found in sensor values before normalization.')
    if not enabled:
        return out
    with open(scaler_config, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    mean = np.asarray(cfg['scaler_mean'], dtype=np.float32)
    scale = np.asarray(cfg['scaler_scale'], dtype=np.float32)
    normed = (vals - mean) / scale
    if np.isnan(normed).any():
        raise ValueError('NaN produced during normalization.')
    out[SENSOR_COLS] = normed
    return out

def window_dataset(df: pd.DataFrame, window_size: int, step: int) -> Tuple[np.ndarray, Optional[np.ndarray], dict]:
    X_raw = df[SENSOR_COLS].astype(float).values
    y_raw = None
    has_labels = 'activity' in df.columns
    if has_labels:
        unknown = set(df['activity'].astype(str).unique()) - set(CLASS_TO_IDX.keys())
        if unknown:
            raise ValueError(f'Unknown labels in activity column: {unknown}')
        y_raw = df['activity'].astype(str).map(CLASS_TO_IDX).values.astype(np.int32)

    X_list, y_list = [], []
    mixed = 0
    purities = []
    for start in range(0, len(X_raw) - window_size + 1, step):
        end = start + window_size
        X_list.append(X_raw[start:end])
        if y_raw is not None:
            binc = np.bincount(y_raw[start:end], minlength=11)
            cls = int(binc.argmax())
            purity = float(binc[cls]) / float(window_size)
            purities.append(purity)
            if purity < 1.0:
                mixed += 1
            y_list.append(cls)

    if not X_list:
        raise ValueError('No windows created. Dataset too small for current WINDOW_SIZE/STEP.')

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int32) if y_list else None
    qc = {
        'n_rows': int(len(df)),
        'n_windows': int(len(X)),
        'has_labels': bool(has_labels),
        'mixed_windows': int(mixed) if has_labels else None,
        'mixed_window_pct': (100.0 * mixed / len(X)) if has_labels else None,
        'mean_majority_purity': float(np.mean(purities)) if has_labels and purities else None,
    }
    return X, y, qc


In [ ]:
# CELL 4: Run evaluation loop
# For each dataset and model: preprocess, infer, compute metrics, and collect results.
def run_model_on_dataset(model_path: Path, X: np.ndarray, y: Optional[np.ndarray]) -> dict:
    model = tf.keras.models.load_model(str(model_path), compile=False)
    probs = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    preds = np.argmax(probs, axis=1)
    conf = np.max(probs, axis=1)
    counts = np.bincount(preds, minlength=11)
    dominant_idx = int(counts.argmax())
    dominant_pct = float(counts[dominant_idx]) * 100.0 / len(preds)

    out = {
        'n_predictions': int(len(preds)),
        'mean_confidence': float(conf.mean()),
        'std_confidence': float(conf.std()),
        'min_confidence': float(conf.min()),
        'max_confidence': float(conf.max()),
        'dominant_class': LABEL_MAP[dominant_idx],
        'dominant_pct': dominant_pct,
        'dead_classes': int(np.sum(counts == 0)),
        'n_predicted_classes': int(np.sum(counts > 0)),
        'pred_counts': {LABEL_MAP[i]: int(counts[i]) for i in range(11)},
        'pred_pct': {LABEL_MAP[i]: float(counts[i]) * 100.0 / len(preds) for i in range(11)},
    }

    if y is not None:
        out['accuracy'] = float(accuracy_score(y, preds))
        out['f1_macro'] = float(f1_score(y, preds, average='macro', zero_division=0))
        out['classification_report'] = classification_report(
            y,
            preds,
            labels=list(range(11)),
            target_names=[LABEL_MAP[i] for i in range(11)],
            zero_division=0,
            output_dict=True,
        )

    return out


all_rows = []
all_class_rows = []

for ds_name, ds_path in DATASETS.items():
    print(f'\n=== DATASET: {ds_name} ===')
    df = pd.read_csv(ds_path)
    df = normalize_schema(df)
    df, unit_mode = maybe_unit_convert_accel(df)
    df = apply_zscore(df, SCALER_CONFIG, ENABLE_NORMALIZATION)
    X, y, qc = window_dataset(df, WINDOW_SIZE, WINDOW_STEP)

    print(f"Rows={qc['n_rows']}, Windows={qc['n_windows']}, Labels={qc['has_labels']}, UnitMode={unit_mode}")
    if qc['has_labels']:
        print(f"MixedWindows={qc['mixed_windows']} ({qc['mixed_window_pct']:.2f}%), MeanPurity={qc['mean_majority_purity']:.4f}")

    for model_name, model_path in MODELS.items():
        if not model_path.exists():
            print(f'  [SKIP] {model_name}: model file not found')
            continue
        try:
            res = run_model_on_dataset(model_path, X, y)
            row = {
                'dataset': ds_name,
                'dataset_path': str(ds_path),
                'model': model_name,
                'model_path': str(model_path),
                'has_labels': qc['has_labels'],
                'unit_mode': unit_mode,
                'normalization': ENABLE_NORMALIZATION,
                'window_size': WINDOW_SIZE,
                'window_step': WINDOW_STEP,
                'n_rows': qc['n_rows'],
                'n_windows': qc['n_windows'],
                'mixed_window_pct': qc['mixed_window_pct'],
                'mean_majority_purity': qc['mean_majority_purity'],
                'n_predictions': res['n_predictions'],
                'mean_confidence': res['mean_confidence'],
                'std_confidence': res['std_confidence'],
                'dominant_class': res['dominant_class'],
                'dominant_pct': res['dominant_pct'],
                'dead_classes': res['dead_classes'],
                'n_predicted_classes': res['n_predicted_classes'],
                'accuracy': res.get('accuracy'),
                'f1_macro': res.get('f1_macro'),
            }
            all_rows.append(row)

            for cls_name, cls_pct in res['pred_pct'].items():
                all_class_rows.append({
                    'dataset': ds_name,
                    'model': model_name,
                    'class_name': cls_name,
                    'pred_pct': cls_pct,
                    'pred_count': res['pred_counts'][cls_name],
                })

            print(f"  [OK] {model_name}: dom={res['dominant_class']} ({res['dominant_pct']:.1f}%), conf={res['mean_confidence']:.3f}")
        except Exception as ex:
            print(f'  [ERROR] {model_name}: {ex}')

summary_df = pd.DataFrame(all_rows)
class_df = pd.DataFrame(all_class_rows)

print('\nDone.')
print('Summary rows:', len(summary_df))
print('Class rows:', len(class_df))

In [ ]:
# CELL 5: Display summaries
# Shows model-level summary table and top predicted classes per dataset/model.
if not summary_df.empty:
    display_cols = [
        'dataset', 'model', 'has_labels', 'normalization',
        'accuracy', 'f1_macro', 'mean_confidence', 'dominant_class', 'dominant_pct', 'n_predicted_classes', 'dead_classes',
        'mixed_window_pct', 'mean_majority_purity'
    ]
    display(summary_df.sort_values(['dataset', 'accuracy', 'f1_macro'], ascending=[True, False, False])[display_cols])

if not class_df.empty:
    top_class_df = (
        class_df.sort_values(['dataset', 'model', 'pred_pct'], ascending=[True, True, False])
                .groupby(['dataset', 'model'], as_index=False)
                .head(3)
    )
    display(top_class_df)

In [ ]:
# CELL 6: Save outputs
# Writes summary and per-class prediction tables to reports/notebook_model_diagnostics.
# Optional: save results
OUT_DIR = PROJECT_ROOT / 'reports' / 'notebook_model_diagnostics'
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUT_DIR / 'summary.csv'
class_path = OUT_DIR / 'per_class_predictions.csv'

summary_df.to_csv(summary_path, index=False)
class_df.to_csv(class_path, index=False)

print('Saved:')
print(' -', summary_path)
print(' -', class_path)